# Nelson-Siegel-Svensson (NSS) Yield Curve Model

This notebook provides a concise overview of the **NSS model** used for constructing the Zero Coupon Yield Curve (ZCYC) for Indian Government Securities, along with the core functions used in the dashboard.

## 1. What is a Yield Curve?

A **yield curve** plots interest rates of bonds with equal credit quality but different maturities. For government securities, the **ZCYC** represents the term structure of interest rates — the pure relationship between time-to-maturity and yield, stripped of coupon effects.

**Why it matters:**
- Benchmarks pricing for all fixed-income instruments
- Signals monetary policy expectations
- Enables fair valuation of any future cash flow

## 2. The Nelson-Siegel Model (1987)

Nelson & Siegel proposed a parsimonious 3-factor model:

$$r(m) = \beta_0 + \beta_1 \left(\frac{1 - e^{-m/\tau}}{m/\tau}\right) + \beta_2 \left(\frac{1 - e^{-m/\tau}}{m/\tau} - e^{-m/\tau}\right)$$

| Parameter | Interpretation |
|-----------|----------------|
| **β₀** | Long-run level (asymptotic rate as m → ∞) |
| **β₁** | Short-term slope; β₀ + β₁ = instantaneous short rate |
| **β₂** | Medium-term hump/trough at maturity ≈ τ |
| **τ** | Decay speed — controls where the hump peaks |

## 3. The Svensson Extension (1994)

Svensson added a **second curvature term** (β₃, τ₂) for greater flexibility:

$$r(m) = \beta_0 + \beta_1 \left(\frac{1 - e^{-m/\tau_1}}{m/\tau_1}\right) + \beta_2 \left(\frac{1 - e^{-m/\tau_1}}{m/\tau_1} - e^{-m/\tau_1}\right) + \beta_3 \left(\frac{1 - e^{-m/\tau_2}}{m/\tau_2} - e^{-m/\tau_2}\right)$$

This **6-parameter** model is used by FBIL/CCIL for the official Indian G-Sec ZCYC.

## 4. Parameter Guide

| Parameter | Role | Typical Effect |
|-----------|------|----------------|
| **β₀** | Long-run level | Shifts entire curve up/down |
| **β₁** | Slope | Negative → normal curve; Positive → inverted |
| **β₂** | 1st curvature | Positive → hump; Negative → trough at τ₁ |
| **β₃** | 2nd curvature | Additional hump/trough at τ₂ |
| **τ₁** | 1st decay | Larger → hump shifts to longer maturities |
| **τ₂** | 2nd decay | Controls second hump location |

---
## 5. Core Functions

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

### 5.1 NSS Rate Function
Computes the Nelson-Siegel-Svensson yield for a given maturity array.

In [2]:
def nss_rate(m: np.ndarray, beta0: float, beta1: float,
             beta2: float, beta3: float, tau1: float, tau2: float) -> np.ndarray:
    """Compute the Nelson-Siegel-Svensson yield for maturity array *m* (in years)."""
    m = np.where(m == 0, 1e-6, m)  # avoid division by zero

    x1 = m / tau1
    x2 = m / tau2

    term1 = (1 - np.exp(-x1)) / x1          # slope factor
    term2 = term1 - np.exp(-x1)              # 1st curvature factor
    term3 = (1 - np.exp(-x2)) / x2 - np.exp(-x2)  # 2nd curvature factor (Svensson)

    return beta0 + beta1 * term1 + beta2 * term2 + beta3 * term3

### 5.2 Load Data
Load the official ZCYC rates and NSS parameters from the Excel files.

In [3]:
DATA_DIR = Path(".")

def load_rates():
    """Load the official ZCYC rates from 0_rates_india.xlsx."""
    df = pd.read_excel(DATA_DIR / "0_rates_india.xlsx")
    df.columns = ["Maturity", "Zero_Coupon_Rate"]
    return df

def load_params():
    """Load NSS parameters from ZCYC_param.xlsx and normalise column names."""
    df = pd.read_excel(DATA_DIR / "ZCYC_param.xlsx")
    cols = df.columns.tolist()
    rename_map = {}
    for c in cols:
        cl = c.lower().strip()
        if "0" in cl and "date" not in cl: rename_map[c] = "beta0"
        elif "1" in cl and "tau" not in cl: rename_map[c] = "beta1"
        elif "2" in cl and "tau" not in cl: rename_map[c] = "beta2"
        elif "3" in cl and "tau" not in cl: rename_map[c] = "beta3"
        elif "tau1" in cl or ("tau" in cl and "1" in cl): rename_map[c] = "tau1"
        elif "tau2" in cl or ("tau" in cl and "2" in cl): rename_map[c] = "tau2"
        elif "date" in cl: rename_map[c] = "date"
    return df.rename(columns=rename_map)

rates_df = load_rates()
params_df = load_params()

print("Rates shape:", rates_df.shape)
display(rates_df.head())
print("\nNSS Parameters:")
display(params_df)

Rates shape: (101, 2)


,Maturity,Zero_Coupon_Rate
0,0.0,5.24
1,0.5,5.35
2,1.0,5.46
3,1.5,5.57
4,2.0,5.67



NSS Parameters:


,date,beta0,beta1,beta2,beta3,tau1,tau2
0,27-02-2026,7.9545,-2.7151,-15.3085,18.7033,18.7118,16.3971


### 5.3 Compute & Plot the Yield Curve
Generate both the original and model-fitted curves on a single interactive chart.

In [4]:
# Extract default parameters
p = params_df.iloc[0]
beta0, beta1, beta2 = float(p["beta0"]), float(p["beta1"]), float(p["beta2"])
beta3, tau1, tau2   = float(p["beta3"]), float(p["tau1"]),  float(p["tau2"])

maturities = rates_df["Maturity"].values
fitted_rates = nss_rate(maturities, beta0, beta1, beta2, beta3, tau1, tau2)

print(f"Parameters → β0={beta0}, β1={beta1}, β2={beta2}, β3={beta3}, τ1={tau1}, τ2={tau2}")

Parameters → β0=7.9545, β1=-2.7151, β2=-15.3085, β3=18.7033, τ1=18.7118, τ2=16.3971


In [5]:
def plot_yield_curves(maturities, original_rates, fitted_rates):
    """Plot original vs NSS-fitted yield curves using Plotly."""
    fig = go.Figure()

    # Original curve
    fig.add_trace(go.Scatter(
        x=maturities, y=original_rates,
        mode="lines", name="Original ZCYC (FBIL/CCIL)",
        line=dict(color="#00d4aa", width=3),
    ))

    # Fitted curve
    fig.add_trace(go.Scatter(
        x=maturities, y=fitted_rates,
        mode="lines", name="NSS Fitted Curve",
        line=dict(color="#ff6b6b", width=2.5, dash="dash"),
    ))

    fig.update_layout(
        template="plotly_dark",
        title="Zero Coupon Yield Curve — India G-Sec",
        xaxis_title="Maturity (Years)",
        yaxis_title="Zero Coupon Rate (%)",
        height=500, hovermode="x unified",
        legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center"),
    )
    return fig

fig = plot_yield_curves(maturities, rates_df["Zero_Coupon_Rate"].values, fitted_rates)
fig.show()

### 5.4 Export Curve Data

In [6]:
def export_curves(maturities, original_rates, fitted_rates, filename="nss_curves.csv"):
    """Export both curves to a CSV file."""
    df = pd.DataFrame({
        "Maturity": maturities,
        "Original_Rate": original_rates,
        "NSS_Fitted_Rate": np.round(fitted_rates, 4),
        "Spread_bps": np.round((fitted_rates - original_rates) * 100, 2),
    })
    df.to_csv(filename, index=False)
    print(f"Exported {len(df)} rows to {filename}")
    return df

export_df = export_curves(maturities, rates_df["Zero_Coupon_Rate"].values, fitted_rates)
display(export_df.head(10))

Exported 101 rows to nss_curves.csv


,Maturity,Original_Rate,NSS_Fitted_Rate,Spread_bps
0,0.0,5.24,5.2394,-0.06
1,0.5,5.35,5.3539,0.39
2,1.0,5.46,5.4636,0.36
3,1.5,5.57,5.5687,-0.13
4,2.0,5.67,5.6694,-0.06
5,2.5,5.77,5.7658,-0.42
6,3.0,5.86,5.8582,-0.18
7,3.5,5.95,5.9467,-0.33
8,4.0,6.03,6.0314,0.14
9,4.5,6.11,6.1124,0.24


---
## 6. References

1. **Nelson, C.R. & Siegel, A.F. (1987)** — *Parsimonious Modeling of Yield Curves*, Journal of Business, 60(4), 473–489. [JSTOR](https://www.jstor.org/stable/2352957)
2. **Svensson, L.E.O. (1994)** — *Estimating and Interpreting Forward Interest Rates: Sweden 1992–1994*, NBER WP 4871. [NBER](https://www.nber.org/papers/w4871)
3. **BIS Papers No. 25 (2005)** — *Zero-Coupon Yield Curves: Technical Documentation*. [BIS](https://www.bis.org/publ/bppdf/bispap25.htm)
4. **FBIL** — Official Indian G-Sec ZCYC methodology. [fbil.org.in](https://www.fbil.org.in/)
5. **Diebold, F.X. & Li, C. (2006)** — *Forecasting the Term Structure of Government Bond Yields*, J. Econometrics. [DOI](https://doi.org/10.1016/j.jeconom.2005.03.005)